In [1]:
%%writefile /kaggle/working/models.py
"""The two architectures under comparison."""
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


class ModelA(nn.Module):
    """Custom scratch CNN. 93,601 params."""

    def __init__(self, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x).squeeze(1)


class ModelB(nn.Module):
    """EfficientNet-B0. 4,008,829 params. pretrained=False is the control arm."""

    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
        self.net = efficientnet_b0(weights=weights)
        self.net.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(1280, 1))

    def forward(self, x):
        return self.net(x).squeeze(1)

    def backbone_parameters(self):
        return self.net.features.parameters()

    def head_parameters(self):
        return self.net.classifier.parameters()

Writing /kaggle/working/models.py


In [2]:
import sys, importlib
sys.path.insert(0, '/kaggle/working')
importlib.invalidate_caches()
import models; importlib.reload(models)
from models import ModelA, ModelB

In [3]:
import os, time
import numpy as np, pandas as pd, torch

R = '/kaggle/input/datasets/rayyanshuda/model-and-dataset-efficiency-results'
SIZE = 160
torch.set_num_threads(4)          # part of the measurement — record it

print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [4]:
def disk_mb(model):
    p = '/kaggle/working/_tmp.pt'
    torch.save(model.state_dict(), p)
    mb = os.path.getsize(p) / 1e6
    os.remove(p)
    return mb

def measure(model, device, bs, n_warm=20, n_iter=100):
    model = model.to(device).eval()
    x = torch.randn(bs, 3, SIZE, SIZE, device=device)
    cuda = device == 'cuda'
    with torch.no_grad():
        for _ in range(n_warm):
            model(x)                                  # warm up, discarded
        if cuda:
            torch.cuda.synchronize()
        ts = []
        for _ in range(n_iter):
            if cuda:
                torch.cuda.synchronize()              # drain prior work
            t0 = time.perf_counter()
            model(x)
            if cuda:
                torch.cuda.synchronize()              # wait for THIS work
            ts.append(time.perf_counter() - t0)
    ts  = np.array(ts) * 1000
    med = float(np.median(ts))
    return {'median_ms': round(med, 3),
            'p95_ms': round(float(np.percentile(ts, 95)), 3),
            'imgs_per_s': round(bs / (med / 1000), 1)}

BUILD = {'A (94k, scratch)':    lambda: ModelA(),
         'B0 (4M, scratch)':    lambda: ModelB(pretrained=False),
         'B0 (4M, pretrained)': lambda: ModelB(pretrained=True)}

rows = []
for name, maker in BUILD.items():
    m = maker()
    params, mb = sum(p.numel() for p in m.parameters()), disk_mb(m)
    for device in (['cuda', 'cpu'] if torch.cuda.is_available() else ['cpu']):
        for bs in (1, 32):
            rows.append({'model': name, 'params': params, 'disk_mb': round(mb, 2),
                         'device': device, 'batch': bs, **measure(maker(), device, bs)})

lat = pd.DataFrame(rows)
lat.to_csv('/kaggle/working/latency.csv', index=False)
print(f'hardware: {torch.cuda.get_device_name(0)} | 4 CPU threads | {SIZE}x{SIZE}\n')
print(lat.to_string(index=False))

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 159MB/s]


hardware: Tesla T4 | 4 CPU threads | 160x160

              model  params  disk_mb device  batch  median_ms  p95_ms  imgs_per_s
   A (94k, scratch)   93601     0.38   cuda      1      0.573   0.641      1745.4
   A (94k, scratch)   93601     0.38   cuda     32      8.402   8.430      3808.8
   A (94k, scratch)   93601     0.38    cpu      1     11.189  12.206        89.4
   A (94k, scratch)   93601     0.38    cpu     32    491.776 516.600        65.1
   B0 (4M, scratch) 4008829    16.31   cuda      1      7.918   8.374       126.3
   B0 (4M, scratch) 4008829    16.31   cuda     32     19.000  19.197      1684.2
   B0 (4M, scratch) 4008829    16.31    cpu      1     25.029  28.844        40.0
   B0 (4M, scratch) 4008829    16.31    cpu     32    499.222 539.467        64.1
B0 (4M, pretrained) 4008829    16.31   cuda      1      8.911  10.054       112.2
B0 (4M, pretrained) 4008829    16.31   cuda     32     19.274  19.489      1660.3
B0 (4M, pretrained) 4008829    16.31    cpu      1  

In [5]:
def epochs_to(prefix, target=0.90):
    out = []
    for s in (0, 1, 2):
        d = pd.read_csv(f'{R}/{prefix}_seed{s}_history.csv')
        hit = d[d.val_auc >= target]
        out.append(int(hit.epoch.iloc[0]) if len(hit) else None)
    return out

for prefix, name in [('model_a', 'A'),
                     ('model_b_scratch', 'B0-scratch'),
                     ('model_b_pretrained', 'B0-pretrained')]:
    print(f'{name:14s} epochs to val AUC 0.90: {epochs_to(prefix)}')

A              epochs to val AUC 0.90: [35, None, None]
B0-scratch     epochs to val AUC 0.90: [16, 13, 13]
B0-pretrained  epochs to val AUC 0.90: [2, 1, 1]


In [6]:
!pip install thop -q
from thop import profile
x = torch.randn(1, 3, 160, 160)
for name, m in [('A', ModelA()), ('B0', ModelB(pretrained=False))]:
    macs, params = profile(m, inputs=(x,), verbose=False)
    print(f'{name:4s} {macs/1e6:8.1f} MMACs  {params/1e6:6.3f} M params')

A       263.8 MMACs   0.094 M params
B0      211.5 MMACs   4.009 M params
